In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np
import datetime, time, warnings
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping
from darts import TimeSeries
from darts.models import BlockRNNModel
from chronos import Chronos2Pipeline
import logging

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
warnings.filterwarnings("ignore", ".*does not have many workers.*")
torch.set_float32_matmul_precision('medium')

if os.path.basename(os.getcwd()) == 'notebooks':
    project_root = os.path.abspath('../..')
else:
    project_root = os.getcwd()

if project_root not in sys.path:
    sys.path.append(project_root)

from src.datamodule import ElectricityDataModule, masked_smoothed_smape
from src.helper import _to_tensor, _extract_context_target_mask, _to_torch, align_forecast_to_target

C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\.venv1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Helper functions loaded successfully


## 1. Setup
 
Define paths and constants for the tuning process.

In [2]:
BASE_DIR = project_root
DATA_DIR = os.path.join(BASE_DIR, "data")
TRAIN_DIR = os.path.join(DATA_DIR, "train_trading_only")
VAL_DIR = os.path.join(DATA_DIR, "val_trading_only")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# --- Tuning Configuration ---
# Use a tiny subset of assets for speed
TUNING_ASSETS = ['Mon05Q1', 'Fri13Q3', 'Sat02Q2']
INPUT_CHUNK_LENGTH = 48
OUTPUT_CHUNK_LENGTH = 10
TARGET_COLS = ["high", "low", "close", "volume"]
SEED = 827

## 2. Data Loading for Tuning

In [3]:
def load_tuning_data(assets, train_dir, val_dir):
    """Loads a small, fixed subset of data for fast tuning."""
    logging.info(f"Loading tuning data for assets: {assets}")
    train_ts, val_ts, future_cov_train, future_cov_val = [], [], [], []

    # Determine covariate columns from the first asset
    first_df = pd.read_parquet(os.path.join(train_dir, f"{assets[0]}.parquet"))
    covariate_cols = [c for c in first_df.columns if c not in TARGET_COLS + ['ExecutionTime', 'is_trading']]
    logging.info(f"Using {len(covariate_cols)} covariates.")

    for asset in assets:
        train_df = pd.read_parquet(os.path.join(train_dir, f"{asset}.parquet")).head(1500)
        val_df = pd.read_parquet(os.path.join(val_dir, f"{asset}.parquet")).head(500)

        for df in [train_df, val_df]:
            df['ExecutionTime'] = pd.to_datetime(df['ExecutionTime'])
            numeric_cols = df.select_dtypes(include=np.number).columns
            df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan).ffill().bfill().fillna(0)
            df[numeric_cols] = df[numeric_cols].astype(np.float32)

        train_ts.append(TimeSeries.from_dataframe(train_df, 'ExecutionTime', TARGET_COLS, freq='15min', fill_missing_dates=True, fillna_value=0))
        val_ts.append(TimeSeries.from_dataframe(val_df, 'ExecutionTime', TARGET_COLS, freq='15min', fill_missing_dates=True, fillna_value=0))
        future_cov_train.append(TimeSeries.from_dataframe(train_df, 'ExecutionTime', covariate_cols, freq='15min', fill_missing_dates=True, fillna_value=0))
        future_cov_val.append(TimeSeries.from_dataframe(val_df, 'ExecutionTime', covariate_cols, freq='15min', fill_missing_dates=True, fillna_value=0))

    return train_ts, val_ts, future_cov_train, future_cov_val

## 3. LSTM Tuning

In [4]:
def fast_lstm_tuning(train_ts, val_ts, future_cov_train, future_cov_val):
    """Performs a quick grid search for LSTM."""
    logging.info("\n" + "="*60 + "\nStarting Fast LSTM Tuning" + "\n" + "="*60)

    param_grid = {
        'hidden_dim': [64, 128],
        'n_rnn_layers': [2, 3],
        'lr': [0.001, 0.005],
        'batch_size': [64, 128],
        'dropout': [0.1, 0.3]
    }

    best_loss = float('inf')
    best_params = {}

    for hidden in param_grid['hidden_dim']:
        for layers in param_grid['n_rnn_layers']:
            for lr in param_grid['lr']:
                for batch in param_grid['batch_size']:
                    for drop in param_grid['dropout']:
                        current_params_for_log = {'hidden_dim': hidden, 'n_rnn_layers': layers, 'dropout': drop, 'lr': lr, 'batch_size': batch}
                        logging.info(f"Testing LSTM with params: {current_params_for_log}")

                        try:
                            model = BlockRNNModel(
                                model="LSTM",
                                input_chunk_length=INPUT_CHUNK_LENGTH,
                                output_chunk_length=OUTPUT_CHUNK_LENGTH,
                                hidden_dim=hidden,
                                n_rnn_layers=layers,
                                dropout=drop,
                                batch_size=batch,
                                optimizer_kwargs={'lr': lr},  # Correctly pass lr
                                loss_fn=torch.nn.L1Loss(),
                                random_state=SEED,
                                force_reset=True,
                                pl_trainer_kwargs={
                                    'accelerator': 'gpu' if torch.cuda.is_available() else 'cpu', 'devices': 1,
                                    'max_epochs': 5, 'enable_progress_bar': True, 'enable_model_summary': False, 'logger': False
                                }
                            )

                            model.fit(
                                series=train_ts, future_covariates=future_cov_train,
                                val_series=val_ts, val_future_covariates=future_cov_val,
                                verbose=False
                            )

                            val_loss = model.trainer.callback_metrics.get("val_loss", torch.tensor(float('inf'))).item()
                            logging.info(f"  -> Val Loss: {val_loss:.4f}")

                            if val_loss < best_loss:
                                best_loss = val_loss
                                best_params = current_params_for_log
                                logging.info(f"  -> New best found!")
                        except Exception as e:
                            logging.error(f"  -> Trial failed: {e}")

    logging.info(f"\nBest LSTM Params: {best_params} (Loss: {best_loss:.4f})")
    return best_params

In [ ]:
# Load data
train_ts, val_ts, future_cov_train, future_cov_val = load_tuning_data(TUNING_ASSETS, TRAIN_DIR, VAL_DIR)

# Run tuning
best_lstm = fast_lstm_tuning(train_ts, val_ts, future_cov_train, future_cov_val)

[INFO] Loading tuning data for assets: ['Mon05Q1', 'Fri13Q3', 'Sat02Q2']
[INFO] Using 26 covariates.
[WARNING] The provided DatetimeIndex was associated with a timezone (tz), which is currently not supported. To avoid unexpected behaviour, the tz information was removed. Consider calling `ts.time_index.tz_localize(UTC)` when exporting the results.To plot the series with the right time steps, consider setting the matplotlib.pyplot `rcParams['timezone']` parameter to automatically convert the time axis back to the original timezone.
[WARNING] The provided DatetimeIndex was associated with a timezone (tz), which is currently not supported. To avoid unexpected behaviour, the tz information was removed. Consider calling `ts.time_index.tz_localize(UTC)` when exporting the results.To plot the series with the right time steps, consider setting the matplotlib.pyplot `rcParams['timezone']` parameter to automatically convert the time axis back to the original timezone.
[WARNING] The provided Date

## 4. Chronos2 Hyperparameter Tuning

In [7]:
def fast_chronos_tuning(val_ts, future_cov_val):
    """Performs a quick check for Chronos params."""
    logging.info("\n" + "="*60 + "\nStarting Fast Chronos Tuning" + "\n" + "="*60)

    param_grid = {
        'quantile_levels': ['small', 'medium'],
        'batch_size': [32, 64],
        'max_context_length': [96, 192],
    }

    best_loss = float('inf')
    best_params = {}

    try:
        pipeline = Chronos2Pipeline.from_pretrained(
            "amazon/chronos-2",
            device_map="cuda" if torch.cuda.is_available() else "cpu"
        )
    except Exception as e:
        logging.error(f"Could not load Chronos model, skipping tuning. Error: {e}")
        return {
            "quantile_levels": "medium", "batch_size": 64, "lr": 0.0003,
            "max_context_length": 96, "actual_quantile_levels": [0.05, 0.25, 0.5, 0.75, 0.95]
        }

    for quantile_level in param_grid['quantile_levels']:
        for batch_size in param_grid['batch_size']:
            for context_len in param_grid['max_context_length']:
                current_params = {'quantile_levels':quantile_level, 'batch_size': batch_size, 'max_context_length': context_len}
                logging.info(f"Testing Chronos with params: {current_params}")

                total_loss = 0
                count = 0

                for i in range(len(val_ts)):
                    series = val_ts[i]
                    covariates = future_cov_val[i]

                    if len(series) < context_len + OUTPUT_CHUNK_LENGTH:
                        continue

                    context = series[:context_len]
                    target = series[context_len:context_len + OUTPUT_CHUNK_LENGTH]

                    forecast = pipeline.predict(context, OUTPUT_CHUNK_LENGTH)
                    median_forecast = torch.quantile(torch.from_numpy(forecast), 0.5, dim=0)

                    loss = torch.nn.functional.l1_loss(torch.from_numpy(target.values()), median_forecast.squeeze(1))
                    total_loss += loss.item()
                    count += 1

                avg_loss = total_loss / count if count > 0 else float('inf')
                logging.info(f"  -> Val Loss: {avg_loss:.4f}")

                if avg_loss < best_loss:
                    best_loss = avg_loss
                    best_params = current_params

    final_params = {
        "actual_quantile_levels": [0.05, 0.25, 0.5, 0.75, 0.95],
        **best_params
    }
    logging.info(f"\nBest Chronos Params: {final_params} (Loss: {best_loss:.4f})")
    return final_params

In [8]:
train_ts, val_ts, future_cov_train, future_cov_val = load_tuning_data(TUNING_ASSETS, TRAIN_DIR, VAL_DIR)

best_chronos = fast_chronos_tuning(val_ts, future_cov_val)

[INFO] Loading tuning data for assets: ['Mon05Q1', 'Fri13Q3', 'Sat02Q2']
[INFO] Using 26 covariates.
[WARNING] The provided DatetimeIndex was associated with a timezone (tz), which is currently not supported. To avoid unexpected behaviour, the tz information was removed. Consider calling `ts.time_index.tz_localize(UTC)` when exporting the results.To plot the series with the right time steps, consider setting the matplotlib.pyplot `rcParams['timezone']` parameter to automatically convert the time axis back to the original timezone.
[WARNING] The provided DatetimeIndex was associated with a timezone (tz), which is currently not supported. To avoid unexpected behaviour, the tz information was removed. Consider calling `ts.time_index.tz_localize(UTC)` when exporting the results.To plot the series with the right time steps, consider setting the matplotlib.pyplot `rcParams['timezone']` parameter to automatically convert the time axis back to the original timezone.
[WARNING] The provided Date

ValueError: Unexpected inputs format

## 5. Save Best Hyperparameters

In [ ]:
# --- Save results ---
logging.info("\n" + "="*60 + "\nSaving Hyperparameters" + "\n" + "="*60)

lstm_path = os.path.join(RESULTS_DIR, "best_params_lstm.json")
with open(lstm_path, 'w') as f:
    json.dump(best_lstm, f, indent=4)
logging.info(f"Saved LSTM parameters to: {lstm_path}")

chronos_path = os.path.join(RESULTS_DIR, "best_params_chronos2.json")
with open(chronos_path, 'w') as f:
    json.dump(best_chronos, f, indent=4)
logging.info(f"Saved Chronos2 parameters to: {chronos_path}")

combined_params = {"LSTM": best_lstm, "chronos": best_chronos}
combined_path = os.path.join(RESULTS_DIR, "best_hyperparameters.json")
with open(combined_path, 'w') as f:
    json.dump(combined_params, f, indent=4)
logging.info(f"Saved combined parameters to: {combined_path}")

print("\n--- Fast Tuning Complete ---")
print("\n--- Best LSTM Parameters ---")
print(json.dumps(best_lstm, indent=4))
print("\n--- Best Chronos2 Parameters ---")
print(json.dumps(best_chronos, indent=4))
